In [ ]:
import geopandas as gpd
import numpy as np
from pystac_client import Client
from shapely import geometry

%matplotlib inline


import os
import sys

sys.path.insert(1, os.path.abspath("../Tools"))  # TODO: What is this?


import joblib

from utils import load_data, predict_xr

In [ ]:
%reload_ext autoreload
%autoreload 2

## Find and load data

Load data and set up your array to use for prediction

In [ ]:
# TODO: Load this shared config from utils for both training and prediction

# Configure some things up front
chunks = {"x": 2048, "y": 2048}
datetime = "2024"

bbox = [
    178.36683199429106,
    -17.706311982643278,
    178.54589121502428,
    -17.561129121023423,
]
bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({"geometry": [bbox_geometry]}, crs="EPSG:4326")
gdf.explore()

In [ ]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

year = "2024"
items = list(
    client.search(collections=["dep_s2_geomad"], bbox=bbox, datetime=year).items()
)

In [ ]:
print(f"Found {len(items)} items in for {datetime}")

In [ ]:
bbox_bounds = [
    178.36574794372646,
    -17.706311982643278,
    178.5471181218782,
    -17.561129121023423,
]

# Run the updated function
data = load_data(
    bbox=bbox_bounds,
    year="2024",  # Pass the target year seamlessly
)

In [ ]:
data

In [ ]:
loaded_model = joblib.load("test_model.dump")

In [ ]:
loaded_data = data.compute()

## Run ML classifier

In [ ]:
from dask.distributed import Client

Client(processes=False)

In [ ]:
filled = loaded_data.fillna(-9999.0)

In [ ]:
feature_cols = [
    "blue",
    "green",
    "red",
    "rededge1",
    "rededge2",
    "rededge3",
    "nir",
    "swir16",
    "swir22",
    "emad",
    "bcmad",
    "smad",
    "ndvi",
    "mndwi",
    "evi",
    "savi",
    "bsi",
    "ndmi",
    "ndbi",
    "ndwi",
    "ndre",
    "elevation",
    "slope",
    "aspect",
    "elevation_std",
    # "mean_vv", "mean_vh", "mean_vv_vh" #to be included after sorting out data fusion / s1_geomad issues
]

# make sure this is still an xarray dataset
filled = filled[feature_cols].copy()

# optional: replace NaNs for prediction
for c in feature_cols:
    filled[c] = filled[c].fillna(0)

In [ ]:
# This runs the actual prediction
# predicted = predict_xr(loaded_model, filled, proba=True)

# In your prediction notebook, make sure clean=True is passed:
predicted = predict_xr(loaded_model, filled, clean=True)

# Into integer
cleaned_predictions = predicted.copy(deep=True)
cleaned_predictions.predictions.data = predicted.predictions.data.astype(np.int8)

cleaned_predictions = cleaned_predictions.rename({"predictions": "lulc"})

# cleaned_predictions = predicted.copy(deep=True)
# cleaned_predictions.predictions.data = predicted.predictions.data.astype(np.int8)
# cleaned_predictions.probabilities.data = predicted.probabilities.data.astype(np.float32)

# cleaned_predictions = cleaned_predictions.rename({"predictions": "lulc", "probabilities": "prob"})

In [ ]:
cleaned_predictions

In [ ]:
from matplotlib import colors

classes = [
    [1, "bare_land", "#968640"],
    [2, "forest", "#064a00"],
    [3, "crops", "#ffce33"],
    [4, "grassland", "#d7ffa0"],
    [5, "settlements", "#b3b2ae"],
    [6, "mangroves", "#07b28d"],
    [7, "water", "#71a8ff"],
    [8, "quarry", "#b03a2e"],
]

values_list = [c[0] for c in classes]
color_list = [c[2] for c in classes]

# Build a listed colormap.
c_map = colors.ListedColormap(color_list)
bounds = values_list + [9]
norm = colors.BoundaryNorm(bounds, c_map.N)

cleaned_predictions.lulc.plot.imshow(cmap=c_map, norm=norm, size=10)

In [ ]:
# Write GeoTIFF
cleaned_predictions.lulc.odc.write_cog("lulc_2024.tif", overwrite=True)